> forward pass (next token logits) vs. decoding phase (token decision)

* 草稿小模型 (Draft Model) & 目标大模型 (Target Model)
* (draft model) tokens speculation
    * forward pass + auto-regressive decoding (token by token)
        * token decision: sampling strategy (argmax, top-k, top-p)
    * ar 是串行，串行生成
* (target model) **parallel** verification
    * forward pass: 1 次
    * 并行验证
* rejection sampling

概念 1：生成过程的两步走 (Forward pass vs. Decoding phase)
在大模型吐出每一个字时，其实都在经历两个阶段：

Forward pass (算概率)：模型经过一顿复杂的数学运算，得出一个词表概率分布（比如：觉得下一个词是“吃”的概率是 80%，“打”的概率是 15%）。

Decoding phase (做决定)：程序要根据上面的概率，真正拍板决定选哪个词。这里 PPT 提到了三种常见的策略：

argmax：贪心策略。永远选概率最高的那一个（选“吃”）。这样生成的文本最稳定，但也最死板。

top-k / top-p：随机抽样策略。不一定选最高的，而是在概率靠前的一小撮候选词里面随机挑一个。这样能让 AI 说话更有创造性和多样性。

概念 2：草稿小模型 (实习生起草)
Draft Model (小模型)：参数量很小、算得很快，但是脑子不太聪明的模型。

Tokens speculation (推测生成)：小模型采用传统的自回归（AR）方式，老老实实地“串行”生成接下来的几个词（比如一口气猜了 4 个词：我 -> 喜 -> 欢 -> 吃）。因为它小，所以即使是一个字一个字蹦，速度也极快。

概念 3：目标大模型 (老板并行审批)
Target Model (大模型)：参数量巨大、算得极慢，但是极其聪明、回答质量极高的模型（也就是我们真正想用的那个模型）。

Parallel verification (并行验证)：这就是最巧妙的地方！老板（大模型）不再自己一个字一个字去写了。它把实习生（小模型）刚刚写好的 4 个词 [我, 喜, 欢, 吃] 一次性打包接过来。

Forward pass 1次：利用我们在前两张图学过的原理，大模型可以通过一次前向传播（配合 Causal Mask），瞬间并行计算出这 4 个词在它眼里的正确概率。

概念 4：拒绝采样 (Rejection Sampling - 纠错机制)
老板看完草稿后，要进行批改：

如果大模型觉得小模型猜的 我、喜、欢 都很完美（概率吻合），就直接采纳（Accept）。

如果大模型算出来，在 喜欢 后面，正确的词应该是 喝，而不是小模型猜的 吃。那么大模型就会在这里触发 拒绝采样 (Rejection sampling)：

保留前面的 我喜欢。

扔掉后面的 吃。

大模型亲自把 喝 补上。

然后让小模型顺着 我喜欢喝 继续去写下一段草稿。

🌟 终极总结：
为什么这样做能提速？
因为对于大模型来说，“做一道 1 个字的填空题”和“做一道 4 个字的判断题”，在 GPU 上的计算时间几乎是一模一样的（因为可以并行计算）。
通过让小模型快速猜（推测），大模型批量改（验证），我们既保证了最终输出的质量100%等同于大模型，又极大地白嫖了小模型的生成速度。这就是目前加速大模型推理的终极魔法之一。